# Canonical GHZ with measurement 0 fixed: shifted-Halton search

This is the requested next experiment from the previous session. It uses the **fixed canonical GHZ state**, fixes input-0 observables to **Z for all three parties**, and covers the six coordinates of the remaining Bloch directions using a reproducible shifted Halton sequence. No inequality or optimized quantum realization from an earlier run is used to generate or select a target.

For every target the notebook solves both established separator LPs, reconstructs exact integer inequalities, and certifies validity and face dimension for the corrected two-orientation cycle polytope. It retains facets and nonfacets. After the batch, it groups all candidates under the full 3,072 physical relabelings and ranks each family by its best critical white-noise fraction over **every stored target and every physical copy**.

The default research batch is **10,000 targets / 20,000 LPs**. For an installation check change only `N_SAMPLES = 4` below; the automatic output name then keeps that test separate from the 10,000-target run. Use a Julia kernel and run cells in order. The notebook is self-contained; the optional companion `.jl` file is the same executable source.

Portable results are written under `C:\ResearchRuns` on Windows, outside OneDrive. Set `RESULTS_ROOT` to another directory if desired. A matching checkpoint is resumed; incompatible or orphaned results are never overwritten. The long research batch is intended to be run locally by you.

## 1. Install the Julia dependencies if needed

`LinearAlgebra`, `Random`, `Printf`, `Dates`, `Serialization`, and `SHA` are Julia standard libraries. The only external packages are `JuMP`, `MathOptInterface`, and `GLPK`.

In [3]:
import Pkg

required_packages = ["JuMP", "MathOptInterface", "GLPK"]
missing_packages = filter(
    package -> Base.find_package(package) === nothing,
    required_packages,
)

if !isempty(missing_packages)
    println("Installing missing packages: ", join(missing_packages, ", "))
    Pkg.add(missing_packages)
else
    println("All required external Julia packages are already installed.")
end

All required external Julia packages are already installed.


## 2. Configuration and conventions

Inputs and outcomes are 0/1. Outcome 0 is the +1 eigenvalue of its observable. Thus the input-0 **observable** is Z, and its two effects are diag(1,0) and diag(0,1). The canonical vector index is

$$j(a,b,c,x,y,z)=1+c+2b+4a+8z+16y+32x.$$

Keep this configuration unchanged when resuming. Changing N_SAMPLES automatically changes the default output name. To repeat an identical batch from scratch, choose a new output directory. The sampler's seed controls one six-component random shift; points themselves are addressed by their sequence index, so interruptions do not alter later targets.

In [5]:
using LinearAlgebra
using Random
using Printf
using Dates
using Serialization
using SHA
using JuMP
using GLPK

# Import MathOptInterface directly. This avoids the JuMP.MathOptInterface
# namespace error encountered with the user's Julia 1.12 installation.
import MathOptInterface as MOI

const TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

state_index(a, b, c) = 1 + c + 2*b + 4*a

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)
@assert [state_index(a, b, c) for (a, b, c) in TRIPLES] == collect(1:8)


# -------------------------- USER CONFIGURATION --------------------------
N_SAMPLES = 10_000                     # use 4 for the installation check
MASTER_SEED = 20261020                 # seed for the six Halton shifts
HALTON_START_INDEX = 1                 # positive; sample i uses start+i-1
RESULTS_ROOT = Sys.iswindows() ? raw"C:\ResearchRuns" : pwd()
SEARCH_OUTPUT_DIRECTORY = joinpath(RESULTS_ROOT,
    "cycle_canonical_ghz_halton_seed$(MASTER_SEED)_start$(HALTON_START_INDEX)_n$(N_SAMPLES)")
RESUME_SEARCH = true
STOP_ON_ERROR = true
CHECKPOINT_EVERY = 100
WRITE_TABLES_EVERY = 500
PRINT_EVERY = 100
CROSS_EVALUATION_BLOCK_SIZE = 256
RANKING_TIE_TOLERANCE = 1e-9            # in normalized gap G
RATIONAL_RECONSTRUCTION_TOLERANCE = 1e-8
LP_CONSISTENCY_TOLERANCE = 2e-7
QUANTUM_AUDIT_TOLERANCE = 2e-11
NORMALIZED_VIOLATION_TOLERANCE = 1e-9
# ------------------------------------------------------------------------
@assert N_SAMPLES>0 && MASTER_SEED>=0 && HALTON_START_INDEX>=1
@assert CHECKPOINT_EVERY>=1 && WRITE_TABLES_EVERY>=1 && PRINT_EVERY>=1
@assert CROSS_EVALUATION_BLOCK_SIZE>=1
BLAS.set_num_threads(min(4,Sys.CPU_THREADS))
println("Planned targets: ",N_SAMPLES,"; planned LPs: ",2N_SAMPLES)
println("Output directory: ",abspath(SEARCH_OUTPUT_DIRECTORY))

Planned targets: 10000; planned LPs: 20000
Output directory: C:\ResearchRuns\cycle_canonical_ghz_halton_seed20261020_start1_n10000


## 3. Generate the corrected cycle/siglocal polytope

Each deterministic strategy receives the two inputs available along one orientation of the communication cycle.

- clockwise: $a=f_A(x,z),\ b=f_B(y,x),\ c=f_C(z,y)$;
- counterclockwise: $a=f_A(x,y),\ b=f_B(y,z),\ c=f_C(z,x)$.

There are 4,096 strategies of each orientation. Their intersection consists of the 64 local deterministic behaviours, so the union has $4096+4096-64=8128$ distinct vertices.

In [7]:
"""Evaluate one of the 16 deterministic binary functions of two bits."""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""Construct one deterministic 64-component cycle behaviour."""
function deterministic_vertex(
    fA::Int,
    fB::Int,
    fC::Int,
    orientation::Symbol,
)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

"""Generate the 4096 deterministic vertices of one orientation."""
function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1

    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end

    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64
    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)
        for context in 0:7
            block = (8*context + 1):(8*context + 8)
            @assert sum(vertex[block]) == 1
        end
    end
    return true
end

validate_deterministic_vertices (generic function with 1 method)

In [8]:
vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)

# The 64 local deterministic behaviours occur in both orientations. Removing
# duplicates does not change the convex hull, but it exposes the true count.
vertices_cycle = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_cycle)

@assert size(vertices_clockwise) == (4096, 64)
@assert size(vertices_counterclockwise) == (4096, 64)
@assert length(intersect(clockwise_set, counterclockwise_set)) == 64
@assert size(vertices_cycle) == (8128, 64)

const UNIFORM_BEHAVIOUR = fill(1//8, 64)

# Every probability coordinate occurs in exactly 1016 vertices. Consequently,
# the centroid of the 8128 distinct vertices is exactly white noise.
@assert all(sum(vertices_cycle; dims=1) .== 1016)
@assert [
    sum(Int(vertices_cycle[i, j]) for i in axes(vertices_cycle, 1)) //
    size(vertices_cycle, 1)
    for j in axes(vertices_cycle, 2)
] == UNIFORM_BEHAVIOUR

println("Corrected cycle polytope")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  intersection:              ", length(intersect(clockwise_set, counterclockwise_set)))
println("  distinct union:            ", size(vertices_cycle, 1))

Corrected cycle polytope
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  intersection:              64
  distinct union:            8128


## 4. Exact dimension and facet certification

Facet status is decided in the 50-dimensional affine hull of the polytope. The routines below compute ranks modulo several large primes and use a Hadamard bound when needed; they do not infer facet status from a floating-point rank alone.

In [10]:
# Pairwise-distinct primes fitting in Int64. Int128 is used for products.
const RANK_PRIMES = Int64[
    1_000_000_007,
    1_000_000_009,
    998_244_353,
    1_004_535_809,
]

mod_multiply(a::Int64, b::Int64, modulus::Int64) =
    Int64(mod(Int128(a) * Int128(b), Int128(modulus)))

function mod_power(base::Int64, exponent::Int64, modulus::Int64)
    result = Int64(1)
    factor = mod(base, modulus)
    power = exponent
    while power > 0
        if isodd(power)
            result = mod_multiply(result, factor, modulus)
        end
        factor = mod_multiply(factor, factor, modulus)
        power >>= 1
    end
    return result
end

"""Rank of an integer matrix over the prime field Z/pZ."""
function rank_mod_prime(
    matrix::AbstractMatrix{<:Integer},
    modulus::Int64,
)
    n_rows, n_columns = size(matrix)
    work = Matrix{Int64}(undef, n_rows, n_columns)

    @inbounds for index in eachindex(matrix)
        work[index] = mod(Int64(matrix[index]), modulus)
    end

    pivot_row = 1
    @inbounds for column in 1:n_columns
        pivot_row > n_rows && break

        pivot = 0
        for row in pivot_row:n_rows
            if work[row, column] != 0
                pivot = row
                break
            end
        end
        pivot == 0 && continue

        if pivot != pivot_row
            for j in 1:n_columns
                work[pivot_row, j], work[pivot, j] =
                    work[pivot, j], work[pivot_row, j]
            end
        end

        inverse_pivot = mod_power(work[pivot_row, column], modulus - 2, modulus)

        for row in (pivot_row + 1):n_rows
            entry = work[row, column]
            entry == 0 && continue
            factor = mod_multiply(entry, inverse_pivot, modulus)
            for j in column:n_columns
                product = mod_multiply(factor, work[pivot_row, j], modulus)
                work[row, j] = Int64(mod(
                    Int128(work[row, j]) - Int128(product),
                    Int128(modulus),
                ))
            end
        end
        pivot_row += 1
    end

    return pivot_row - 1
end

"""Certify the rational rank of an integer matrix."""
function certified_integer_rank(matrix::AbstractMatrix{<:Integer})
    n_rows, n_columns = size(matrix)
    min(n_rows, n_columns) == 0 && return 0

    ranks = [rank_mod_prime(matrix, prime) for prime in RANK_PRIMES]
    certified_rank = maximum(ranks)
    certified_rank == min(n_rows, n_columns) && return certified_rank

    maximum_squared_row_norm = maximum(
        sum(BigInt(matrix[row, column])^2 for column in 1:n_columns)
        for row in 1:n_rows
    )
    maximum_squared_row_norm == 0 && return 0

    next_minor_size = certified_rank + 1
    modulus_product = prod(BigInt(prime) for prime in RANK_PRIMES)
    hadamard_squared_bound = maximum_squared_row_norm^next_minor_size

    modulus_product^2 > hadamard_squared_bound || error(
        "The modular primes are insufficient to certify this rank exactly.",
    )
    return certified_rank
end

function difference_matrix(points::AbstractMatrix{<:Integer})
    n_points, dimension = size(points)
    n_points <= 1 && return zeros(Int64, 0, dimension)
    differences = Matrix{Int64}(undef, n_points - 1, dimension)
    @inbounds for row in 2:n_points, column in 1:dimension
        differences[row - 1, column] =
            Int64(points[row, column]) - Int64(points[1, column])
    end
    return differences
end

function certified_affine_dimension(points::AbstractMatrix{<:Integer})
    n_points = size(points, 1)
    n_points == 0 && return -1
    n_points == 1 && return 0
    return certified_integer_rank(difference_matrix(points))
end

const SIGLOCAL_AFFINE_DIMENSION = certified_affine_dimension(vertices_cycle)
@assert SIGLOCAL_AFFINE_DIMENSION == 50

println("Exact affine dimension: ", SIGLOCAL_AFFINE_DIMENSION)

Exact affine dimension: 50


## 5. Fixed GHZ state and the six-dimensional design

The state is $(|000\rangle+|111\rangle)/\sqrt2$, with phase zero. Use Halton bases (2,3,5,7,11,13), one seed-derived shift in [0,1)^6, and reduce the shifted coordinates modulo one. Pair the coordinates as $(u_A,v_A,u_B,v_B,u_C,v_C)$. For each party,

$$n_z=1-2u,\quad\phi=2\pi v,\quad\theta=\arccos(n_z),\quad
M_1=\sin\theta\cos\phi\,X+\sin\theta\sin\phi\,Y+\cos\theta\,Z.$$

Using uniform u, rather than uniform theta, gives the uniform-area sphere parametrization. Both polar hemispheres and the full azimuth are retained; no outcome or party relabeling is used to prune the sampler. This is a low-discrepancy design, not a collection of independent random targets.

The notebook independently compares tensor-product Born probabilities with the closed GHZ correlator formula for every target. Raw Halton coordinates, angles, Bloch directions, shifts, state amplitudes, effects and probabilities are exported.

In [12]:
const I2 = Matrix{ComplexF64}(I, 2, 2)
const PAULI_X = ComplexF64[0 1; 1 0]
const PAULI_Y = ComplexF64[0 -im; im 0]
const PAULI_Z = ComplexF64[1 0; 0 -1]
const COMPUTATIONAL_MEASUREMENT = [ComplexF64[1 0; 0 0], ComplexF64[0 0; 0 1]]
const CANONICAL_GHZ = ComplexF64[1,0,0,0,0,0,0,1] / sqrt(2)
const HALTON_BASES = [2,3,5,7,11,13]
const HALTON_SHIFT = rand(MersenneTwister(MASTER_SEED), 6)

"""Radical inverse in base b, with the first nonzero sequence index equal to one."""
function radical_inverse(index::Integer, base::Integer)
    index >= 0 && base >= 2 || error("Invalid radical-inverse arguments")
    n=index; weight=1.0/base; result=0.0
    while n>0
        digit=rem(n,base); result+=digit*weight
        n=div(n,base); weight/=base
    end
    result
end

function shifted_halton_point(sequence_index)
    [mod(radical_inverse(sequence_index, HALTON_BASES[j])+HALTON_SHIFT[j],1.0) for j in 1:6]
end

"""Six Halton coordinates: (u_A,v_A,u_B,v_B,u_C,v_C)."""
function canonical_ghz_realization(sequence_index)
    point=shifted_halton_point(sequence_index)
    measurements=Vector{Vector{Vector{Matrix{ComplexF64}}}}(undef,3)
    angles=zeros(Float64,3,2); directions=zeros(Float64,3,3)
    for party in 1:3
        u,v=point[2party-1],point[2party]
        nz=1-2u; phi=2pi*v; transverse=sqrt(max(0.0,1-nz*nz))
        nx=transverse*cos(phi); ny=transverse*sin(phi)
        observable=nx*PAULI_X+ny*PAULI_Y+nz*PAULI_Z
        E0=(I2+observable)/2
        measurements[party]=[deepcopy(COMPUTATIONAL_MEASUREMENT),[E0,I2-E0]]
        angles[party,:]=[acos(clamp(nz,-1.0,1.0)),phi]
        directions[party,:]=[nx,ny,nz]
    end
    copy(CANONICAL_GHZ),measurements,point,angles,directions
end

"""Independent closed-form GHZ probabilities, using correlators rather than tensors."""
function analytic_ghz_behavior(measurements)
    observables=[[measurements[p][x][1]-measurements[p][x][2] for x in 1:2] for p in 1:3]
    directions=[[[real(O[1,2]),-imag(O[1,2]),real(O[1,1])] for O in observables[p]] for p in 1:3]
    q=zeros(64)
    for (x,y,z) in TRIPLES,(a,b,c) in TRIPLES
        A,B,C=directions[1][x+1],directions[2][y+1],directions[3][z+1]
        sa,sb,sc=(-1.0)^a,(-1.0)^b,(-1.0)^c
        triple=real((A[1]+im*A[2])*(B[1]+im*B[2])*(C[1]+im*C[2]))
        q[prob_index(a,b,c,x,y,z)]=(1+sa*sb*A[3]*B[3]+sa*sc*A[3]*C[3]+sb*sc*B[3]*C[3]+sa*sb*sc*triple)/8
    end
    q
end

@assert radical_inverse(1,2)==0.5
@assert radical_inverse(2,2)==0.25
@assert radical_inverse(3,2)==0.75
@assert isapprox(radical_inverse(2,3),2/3;atol=1e-16)
@assert all(0 .<= HALTON_SHIFT .< 1)
println("Shifted Halton bases: ",HALTON_BASES)
println("Recorded shift: ",join([@sprintf("%.17g",x) for x in HALTON_SHIFT],","))

"""Compute p(abc|xyz) from the Born rule in the canonical order."""
function quantum_behavior(state, measurements)
    @assert length(state) == 8
    probabilities = zeros(Float64, 64)
    maximum_imaginary_part = 0.0

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        operator = kron(
            kron(measurements[1][x + 1][a + 1], measurements[2][y + 1][b + 1]),
            measurements[3][z + 1][c + 1],
        )
        expectation = dot(state, operator * state)
        maximum_imaginary_part = max(
            maximum_imaginary_part,
            abs(imag(expectation)),
        )
        probability = real(expectation)
        probabilities[prob_index(a, b, c, x, y, z)] =
            abs(probability) <= 1e-14 ? 0.0 : probability
    end

    return probabilities, maximum_imaginary_part
end

Shifted Halton bases: [2, 3, 5, 7, 11, 13]
Recorded shift: 0.60800001893882372,0.513675690027938,0.4159411545223437,0.92719294482674242,0.64994627745284017,0.17217532967614879


quantum_behavior

## 6. Audit every generated quantum behaviour

Before either LP is solved, the notebook checks the state norm, projector properties, measurement completeness, Born probabilities, normalization, and all two-party nonsignalling marginals. A failed audit stops the default run and writes a failure record.

In [14]:
function nonsignalling_residuals(probabilities)
    normalization = 0.0
    ab = 0.0
    ac = 0.0
    bc = 0.0

    for (x, y, z) in TRIPLES
        block_sum = sum(
            probabilities[prob_index(a, b, c, x, y, z)]
            for (a, b, c) in TRIPLES
        )
        normalization = max(normalization, abs(block_sum - 1))
    end

    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, y, 0)] for c in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, y, 1)] for c in 0:1)
        ab = max(ab, abs(lhs - rhs))
    end

    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, 0, z)] for b in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, 1, z)] for b in 0:1)
        ac = max(ac, abs(lhs - rhs))
    end

    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, 0, y, z)] for a in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, 1, y, z)] for a in 0:1)
        bc = max(bc, abs(lhs - rhs))
    end

    return (normalization=normalization, ab=ab, ac=ac, bc=bc)
end

"""Return diagnostics and throw if a realization is not a valid quantum model."""
function audit_quantum_realization(
    state,
    measurements,
    probabilities,
    maximum_imaginary_part;
    tolerance=QUANTUM_AUDIT_TOLERANCE,
)
    state_norm_error = abs(real(dot(state, state)) - 1)
    maximum_hermiticity_error = 0.0
    maximum_idempotence_error = 0.0
    maximum_completeness_error = 0.0
    minimum_projector_eigenvalue = Inf

    for party in 1:3, input in 1:2
        projectors = measurements[party][input]
        @assert length(projectors) == 2
        maximum_completeness_error = max(
            maximum_completeness_error,
            opnorm(projectors[1] + projectors[2] - I2),
        )

        for projector in projectors
            maximum_hermiticity_error = max(
                maximum_hermiticity_error,
                opnorm(projector - projector'),
            )
            maximum_idempotence_error = max(
                maximum_idempotence_error,
                opnorm(projector * projector - projector),
            )
            minimum_projector_eigenvalue = min(
                minimum_projector_eigenvalue,
                minimum(eigvals(Hermitian(projector))),
            )
        end
    end

    ns = nonsignalling_residuals(probabilities)
    minimum_probability = minimum(probabilities)
    maximum_probability = maximum(probabilities)

    state_norm_error <= tolerance || error("State normalization audit failed")
    maximum_hermiticity_error <= tolerance || error("Hermiticity audit failed")
    maximum_idempotence_error <= tolerance || error("Projectivity audit failed")
    maximum_completeness_error <= tolerance || error("Measurement completeness audit failed")
    minimum_projector_eigenvalue >= -tolerance || error("A projector is not positive")
    maximum_imaginary_part <= tolerance || error("Born probability has a non-negligible imaginary part")
    minimum_probability >= -tolerance || error("A Born probability is negative")
    maximum_probability <= 1 + tolerance || error("A Born probability exceeds one")
    ns.normalization <= tolerance || error("Probability normalization audit failed")
    maximum((ns.ab, ns.ac, ns.bc)) <= tolerance || error("Nonsignalling audit failed")

    return Dict{String, Any}(
        "state_norm_error" => state_norm_error,
        "measurement_hermiticity_error" => maximum_hermiticity_error,
        "measurement_idempotence_error" => maximum_idempotence_error,
        "measurement_completeness_error" => maximum_completeness_error,
        "minimum_projector_eigenvalue" => minimum_projector_eigenvalue,
        "born_imaginary_error" => maximum_imaginary_part,
        "minimum_probability" => minimum_probability,
        "maximum_probability" => maximum_probability,
        "normalization_residual" => ns.normalization,
        "ab_nonsignalling_residual" => ns.ab,
        "ac_nonsignalling_residual" => ns.ac,
        "bc_nonsignalling_residual" => ns.bc,
    )
end

audit_quantum_realization

## 7. Build the two reusable LPs

The models contain all 8,128 corrected vertices. Only their objectives change from one quantum target to the next.

The 64-coordinate representation is retained directly. The free-$\beta$ model therefore has the usual affine-gauge freedom in its coefficient vector. Later deduplication uses the exact slack on every cycle vertex, so coefficient vectors differing only by affine equalities are recognized as the same geometric inequality.

In [16]:
function build_fixed_beta_model(vertices)
    number_of_vertices, dimension = size(vertices)
    model = Model(GLPK.Optimizer)
    set_silent(model)

    @variable(model, coefficients[1:dimension] >= 0)
    @constraint(
        model,
        valid_on_cycle[i=1:number_of_vertices],
        sum(vertices[i, j] * coefficients[j] for j in 1:dimension) >= 1,
    )
    @objective(model, Min, sum(coefficients))
    return model, coefficients
end

function build_free_beta_model(vertices, uniform_behavior)
    number_of_vertices, dimension = size(vertices)
    model = Model(GLPK.Optimizer)
    set_silent(model)

    # Variables are unrestricted unless a bound is supplied.
    @variable(model, coefficients[1:dimension])
    @variable(model, beta)
    @constraint(
        model,
        valid_on_cycle[i=1:number_of_vertices],
        sum(vertices[i, j] * coefficients[j] for j in 1:dimension) >= beta,
    )
    @constraint(
        model,
        white_noise_normalization,
        sum(Float64(uniform_behavior[j]) * coefficients[j] for j in 1:dimension) - beta == 1,
    )
    @objective(model, Min, sum(coefficients) - beta)
    return model, coefficients, beta
end

function solve_fixed_beta_target!(model, coefficients, target)
    set_objective_coefficient(model, coefficients, Float64.(target))
    optimize!(model)
    status = termination_status(model)
    status == MOI.OPTIMAL || error("Fixed-beta LP ended with status $status")
    return (
        coefficients=value.(coefficients),
        beta=1.0,
        solver_objective=objective_value(model),
        status=string(status),
    )
end

function solve_free_beta_target!(model, coefficients, beta, target)
    set_objective_coefficient(model, coefficients, Float64.(target))
    set_objective_coefficient(model, beta, -1.0)
    optimize!(model)
    status = termination_status(model)
    status == MOI.OPTIMAL || error("Free-beta LP ended with status $status")
    return (
        coefficients=value.(coefficients),
        beta=value(beta),
        solver_objective=objective_value(model),
        status=string(status),
    )
end

fixed_beta_model, fixed_beta_coefficients =
    build_fixed_beta_model(vertices_cycle)

free_beta_model, free_beta_coefficients, free_beta_beta =
    build_free_beta_model(vertices_cycle, UNIFORM_BEHAVIOUR)

println("Both reusable LP models were built successfully.")

Both reusable LP models were built successfully.


## 8. Exact reconstruction, validity, facet status, and robustness

GLPK returns floating-point coefficient vectors. For every solution the notebook:

1. rationally reconstructs all coefficients (and $\beta$ in the free LP);
2. clears denominators;
3. computes the **exact** minimum over all 8,128 vertices and uses it as the certified bound $B$;
4. requires that the reconstructed target objective agrees with the LP objective;
5. reduces $(C,B)$ to primitive integers;
6. evaluates the quantum target and white noise;
7. certifies the dimension of the saturating face.

For $C\cdot p\ge B$, define

$$
\Delta=B-C\cdot q,\qquad
W=C\cdot u-B,\qquad
G=\frac{\Delta}{W}.
$$

When $\Delta>0$, the white-noise fraction tolerated by this particular target is

$$
\eta=\frac{G}{1+G}.
$$

These quantities are invariant under positive rescaling and under affine-gauge changes on the polytope.

In [18]:
function rationalize_scalar(value; tolerance=RATIONAL_RECONSTRUCTION_TOLERANCE)
    isfinite(value) || error("Cannot rationalize a non-finite LP coefficient")
    return abs(value) <= tolerance ?
        BigInt(0)//BigInt(1) :
        rationalize(BigInt, value; tol=tolerance)
end

function rationalize_solution(coefficients_float, beta_float)
    coefficients = Rational{BigInt}[
        rationalize_scalar(value) for value in coefficients_float
    ]
    beta = rationalize_scalar(beta_float)
    return coefficients, beta
end

"""Clear denominators simultaneously from coefficients and beta."""
function integer_form(coefficients, beta)
    all_values = vcat(coefficients, [beta])
    common_denominator = foldl(
        lcm,
        denominator.(all_values);
        init=BigInt(1),
    )
    integers = BigInt[
        numerator(value * common_denominator) for value in all_values
    ]
    return integers[1:end-1], integers[end], common_denominator
end

"""Evaluate an integer coefficient vector on every deterministic vertex."""
function integer_vertex_values(vertices, coefficients)
    number_of_vertices, dimension = size(vertices)
    @assert length(coefficients) == dimension
    values = Vector{BigInt}(undef, number_of_vertices)

    @inbounds for row in 1:number_of_vertices
        total = BigInt(0)
        for column in 1:dimension
            if vertices[row, column] != 0
                total += coefficients[column]
            end
        end
        values[row] = total
    end
    return values
end

function primitive_integer_inequality(coefficients, bound)
    common_divisor = foldl(
        gcd,
        abs.(vcat(coefficients, [bound]));
        init=BigInt(0),
    )
    common_divisor == 0 && error("The reconstructed inequality is zero")
    return coefficients .÷ common_divisor, bound ÷ common_divisor
end

function relative_agreement(actual, expected; tolerance=LP_CONSISTENCY_TOLERANCE)
    return abs(actual - expected) <= tolerance * (1 + abs(expected))
end

"""
Return an exact, gauge- and scale-invariant key from all vertex slacks.
The full string is used as the dictionary key; SHA-256 is only a readable ID.
"""
function canonical_slack_key(vertex_values, bound)
    slacks = vertex_values .- bound
    minimum(slacks) == 0 || error("A certified bound must be saturated")
    minimum(slacks) >= 0 || error("The reconstructed inequality is invalid")

    nonzero_slacks = filter(value -> !iszero(value), slacks)
    isempty(nonzero_slacks) && return nothing, nothing
    divisor = foldl(gcd, abs.(nonzero_slacks); init=BigInt(0))
    canonical_slacks = slacks .÷ divisor
    key = join(string.(canonical_slacks), ",")
    identifier = bytes2hex(sha256(codeunits(key)))
    return key, identifier
end

"""Exact affine dimension of a proper face; facet detection can stop at 49."""
function certified_face_dimension(points)
    size(points, 1) == 0 && return -1
    size(points, 1) == 1 && return 0
    differences = difference_matrix(points)

    # Because the inequality is valid, tight, and nonconstant on a
    # 50-dimensional polytope, this face has dimension at most 49. Witnessing
    # rank 49 modulo one prime therefore certifies a facet immediately.
    first_rank = rank_mod_prime(differences, first(RANK_PRIMES))
    first_rank == SIGLOCAL_AFFINE_DIMENSION - 1 && return first_rank

    return certified_integer_rank(differences)
end

# Exact slack keys identify identical faces, so repeated LP outputs reuse a certified rank.
const FACE_AUDIT_CACHE=Dict{String,Int}()

Dict{String, Int64}()

In [19]:
"""Audit one numerical LP solution and return a self-contained outcome."""
function audit_lp_solution(
    lp_kind::String,
    numerical_solution,
    target,
    vertices_cycle,
    vertices_clockwise,
    vertices_counterclockwise,
)
    coefficients_rational, beta_rational = rationalize_solution(
        numerical_solution.coefficients,
        numerical_solution.beta,
    )
    coefficients_raw, beta_raw, common_denominator = integer_form(
        coefficients_rational,
        beta_rational,
    )

    if lp_kind == "fixed_beta"
        minimum(coefficients_raw) >= 0 || error(
            "Fixed-beta reconstruction produced a negative coefficient",
        )
    elseif lp_kind != "free_beta"
        error("Unknown LP kind: $lp_kind")
    end

    cycle_values_raw = integer_vertex_values(vertices_cycle, coefficients_raw)
    exact_bound_raw = minimum(cycle_values_raw)
    reconstructed_bound_float = Float64(exact_bound_raw // common_denominator)

    # Check that rational reconstruction preserves the solver's active bound.
    expected_beta = numerical_solution.beta
    relative_agreement(reconstructed_bound_float, expected_beta) || error(
        "Reconstructed classical bound disagrees with the LP beta",
    )

    reconstructed_target_linear = sum(
        Float64(coefficients_raw[j] // common_denominator) * target[j]
        for j in eachindex(target)
    )
    reconstructed_solver_objective = if lp_kind == "fixed_beta"
        reconstructed_target_linear
    else
        reconstructed_target_linear - Float64(beta_raw // common_denominator)
    end
    relative_agreement(
        reconstructed_solver_objective,
        numerical_solution.solver_objective,
    ) || error("Rational reconstruction changed the LP objective")

    coefficients, bound = primitive_integer_inequality(
        coefficients_raw,
        exact_bound_raw,
    )

    cycle_values = integer_vertex_values(vertices_cycle, coefficients)
    clockwise_values = integer_vertex_values(vertices_clockwise, coefficients)
    counterclockwise_values = integer_vertex_values(vertices_counterclockwise, coefficients)
    exact_bound = minimum(cycle_values)
    exact_bound == bound || error("Primitive reduction changed the bound")
    minimum(cycle_values .- bound) == 0 || error("The exact inequality is not tight")
    minimum(cycle_values .- bound) >= 0 || error("The exact inequality is invalid")

    uniform_slack = sum(coefficients) // 8 - bound
    target_value = sum(Float64(coefficients[j]) * target[j] for j in eachindex(target))
    gap = Float64(bound) - target_value

    # In the fixed-beta LP an interior target can return an affine equality,
    # such as normalization. It is valid but is not a Bell inequality and has
    # no white-noise normalization. Record it as a noncandidate LP outcome.
    if uniform_slack == 0
        all(cycle_values .== bound) || error(
            "Zero uniform slack is inconsistent with the vertex centroid",
        )
        return Dict{String, Any}(
            "lp_kind" => lp_kind,
            "status" => numerical_solution.status,
            "solver_objective" => numerical_solution.solver_objective,
            "outcome_kind" => "affine_equality",
            "candidate" => nothing,
            "target_value" => target_value,
            "gap" => gap,
            "normalized_gap" => NaN,
            "critical_visibility" => NaN,
            "critical_noise_fraction" => NaN,
            "target_violates" => false,
        )
    end

    uniform_slack > 0 || error("The reconstructed inequality has negative white-noise slack")
    normalized_gap = gap / Float64(uniform_slack)
    target_violates = normalized_gap > NORMALIZED_VIOLATION_TOLERANCE
    critical_visibility = target_violates ? 1 / (1 + normalized_gap) : NaN
    critical_noise_fraction = target_violates ? normalized_gap / (1 + normalized_gap) : NaN

    saturating_indices = findall(==(bound), cycle_values)
    isempty(saturating_indices) && error("The reconstructed inequality is not saturated")

    key, candidate_id = canonical_slack_key(cycle_values, bound)
    key === nothing && error("A nonconstant inequality has a constant slack vector")
    face_dimension = get!(FACE_AUDIT_CACHE,key) do
        certified_face_dimension(vertices_cycle[saturating_indices,:])
    end

    candidate = Dict{String, Any}(
        "candidate_id" => candidate_id,
        "canonical_slack_key" => key,
        "coefficients" => coefficients,
        "bound" => bound,
        "uniform_slack" => uniform_slack,
        "cycle_minimum" => minimum(cycle_values),
        "clockwise_minimum" => minimum(clockwise_values),
        "counterclockwise_minimum" => minimum(counterclockwise_values),
        "saturation_count" => length(saturating_indices),
        "face_dimension" => face_dimension,
        "is_facet" => face_dimension == SIGLOCAL_AFFINE_DIMENSION - 1,
    )

    return Dict{String, Any}(
        "lp_kind" => lp_kind,
        "status" => numerical_solution.status,
        "solver_objective" => numerical_solution.solver_objective,
        "outcome_kind" => "inequality",
        "candidate" => candidate,
        "target_value" => target_value,
        "gap" => gap,
        "normalized_gap" => normalized_gap,
        "critical_visibility" => critical_visibility,
        "critical_noise_fraction" => critical_noise_fraction,
        "target_violates" => target_violates,
    )
end

audit_lp_solution

## 9. Persistence and exact candidate records

Both LPs remain in the direct 64-probability representation. The fixed-bound LP includes c>=0 and bound 1; the free-bound LP has unrestricted coefficients and beta, with c.u-beta=1. Exact reconstruction checks every inequality on all 8,128 vertices and both orientations. Affine equalities returned by the fixed-bound LP are logged without treating them as witnesses. Facetness uses certified modular ranks and a Hadamard minor bound; numerical rank alone is not used.

Only a target for which generation, both solves and all audits succeed is committed. The checkpoint is written every CHECKPOINT_EVERY targets, on errors, and in the finalization block. Human-readable tables are written every WRITE_TABLES_EVERY targets and at the end. A hard crash can lose the uncheckpointed suffix; those targets are regenerated deterministically. A normal notebook interruption saves the committed prefix.

All exact candidates are retained, including nonviolated ones. `candidate_ranking.tsv` is a discovery-log ranking; **the final symmetry-completed result is `orbit_ranking.tsv`**. LP degeneracy can change the chosen coefficient representative or facet with solver version, basis state or resume history. The target sequence is independently reproducible; metadata records package versions.

In [21]:
function rational_string(value::Rational)
    denominator(value) == 1 && return string(numerator(value))
    return "$(numerator(value))/$(denominator(value))"
end

float_string(value) = isfinite(Float64(value)) ? @sprintf("%.17g", Float64(value)) : ""
integer_vector_string(values) = join(string.(values), ",")
safe_text(value) = replace(string(value), '\t' => ' ', '\n' => ' ')

function atomic_write(writer, path)
    temporary_path = path * ".tmp"
    open(temporary_path, "w") do io
        writer(io)
    end
    mv(temporary_path, path; force=true)
    return path
end

function write_checkpoint(path, state)
    return atomic_write(path) do io
        serialize(io, state)
    end
end

function read_checkpoint(path)
    return open(path, "r") do io
        deserialize(io)
    end
end

function append_failure(path, sample_id, ensemble, stage, error_value)
    new_file = !isfile(path)
    open(path, "a") do io
        if new_file
            println(io, "timestamp\tsample_id\tensemble\tstage\tmessage")
        end
        println(io, join([
            string(now()),
            string(sample_id),
            string(ensemble),
            string(stage),
            safe_text(sprint(showerror, error_value)),
        ], '\t'))
    end
end

function output_paths(output_directory)
    return Dict(
        "checkpoint" => joinpath(output_directory, "search_checkpoint.jls"),
        "samples" => joinpath(output_directory, "samples.tsv"),
        "lp_runs" => joinpath(output_directory, "lp_runs.tsv"),
        "candidates" => joinpath(output_directory, "unique_candidates.tsv"),
        "ranking" => joinpath(output_directory, "candidate_ranking.tsv"),
        "states" => joinpath(output_directory, "quantum_states.tsv"),
        "measurements" => joinpath(output_directory, "quantum_measurements.tsv"),
        "behaviors" => joinpath(output_directory, "quantum_behaviors.tsv"),
        "failures" => joinpath(output_directory, "failures.tsv"),
        "metadata" => joinpath(output_directory, "run_metadata.txt"),
    )
end

output_paths (generic function with 1 method)

In [22]:
function write_sample_table(path, samples)
    headers = [
        "sample_id", "ensemble", "ensemble_index", "seed",
        "state_norm_error", "measurement_hermiticity_error",
        "measurement_idempotence_error", "measurement_completeness_error",
        "minimum_projector_eigenvalue", "born_imaginary_error",
        "minimum_probability", "maximum_probability", "normalization_residual",
        "ab_nonsignalling_residual", "ac_nonsignalling_residual",
        "bc_nonsignalling_residual", "analytic_ghz_discrepancy", "sequence_index",
    ]
    atomic_write(path) do io
        println(io, join(headers, '\t'))
        for sample in samples
            audit = sample["audit"]
            println(io, join([
                string(sample["sample_id"]), string(sample["ensemble"]),
                string(sample["ensemble_index"]), string(sample["seed"]),
                float_string(audit["state_norm_error"]),
                float_string(audit["measurement_hermiticity_error"]),
                float_string(audit["measurement_idempotence_error"]),
                float_string(audit["measurement_completeness_error"]),
                float_string(audit["minimum_projector_eigenvalue"]),
                float_string(audit["born_imaginary_error"]),
                float_string(audit["minimum_probability"]),
                float_string(audit["maximum_probability"]),
                float_string(audit["normalization_residual"]),
                float_string(audit["ab_nonsignalling_residual"]),
                float_string(audit["ac_nonsignalling_residual"]),
                float_string(audit["bc_nonsignalling_residual"]),
                float_string(audit["analytic_ghz_discrepancy"]),string(sample["sequence_index"]),
            ], '\t'))
        end
    end
end

function write_lp_run_table(path, runs)
    headers = [
        "sample_id", "ensemble", "seed", "lp_kind", "solver_status",
        "solver_objective", "outcome_kind", "candidate_id", "is_duplicate",
        "classical_bound", "target_value", "gap", "uniform_slack",
        "normalized_gap", "target_violates", "critical_visibility",
        "critical_noise_fraction", "saturation_count", "face_dimension", "is_facet",
    ]
    atomic_write(path) do io
        println(io, join(headers, '\t'))
        for run in runs
            println(io, join([
                string(run["sample_id"]), string(run["ensemble"]), string(run["seed"]),
                run["lp_kind"], run["status"], float_string(run["solver_objective"]),
                run["outcome_kind"], run["candidate_id"], string(run["is_duplicate"]),
                run["classical_bound"], float_string(run["target_value"]),
                float_string(run["gap"]), run["uniform_slack"],
                float_string(run["normalized_gap"]), string(run["target_violates"]),
                float_string(run["critical_visibility"]),
                float_string(run["critical_noise_fraction"]),
                string(run["saturation_count"]), string(run["face_dimension"]),
                string(run["is_facet"]),
            ], '\t'))
        end
    end
end

function write_candidate_table(path, candidates)
    headers = [
        "candidate_id", "first_sample_id", "best_sample_id", "lp_kinds",
        "discovery_count", "bound", "uniform_slack", "cycle_minimum",
        "clockwise_minimum", "counterclockwise_minimum", "saturation_count",
        "face_dimension", "is_facet", "best_target_value", "best_gap",
        "best_normalized_gap", "best_critical_visibility",
        "best_critical_noise_fraction", "coefficients_in_canonical_64_order",
    ]
    atomic_write(path) do io
        println(io, join(headers, '\t'))
        for candidate in candidates
            println(io, join([
                candidate["candidate_id"], string(candidate["first_sample_id"]),
                string(candidate["best_sample_id"]), join(candidate["lp_kinds"], ","),
                string(candidate["discovery_count"]), string(candidate["bound"]),
                rational_string(candidate["uniform_slack"]),
                string(candidate["cycle_minimum"]), string(candidate["clockwise_minimum"]),
                string(candidate["counterclockwise_minimum"]),
                string(candidate["saturation_count"]), string(candidate["face_dimension"]),
                string(candidate["is_facet"]), float_string(candidate["best_target_value"]),
                float_string(candidate["best_gap"]), float_string(candidate["best_normalized_gap"]),
                float_string(candidate["best_critical_visibility"]),
                float_string(candidate["best_critical_noise_fraction"]),
                integer_vector_string(candidate["coefficients"]),
            ], '\t'))
        end
    end
end

function write_ranking_table(path, candidates)
    violating = filter(candidate -> candidate["best_normalized_gap"] > NORMALIZED_VIOLATION_TOLERANCE, candidates)
    ranking = sort(violating; by=candidate -> candidate["best_critical_noise_fraction"], rev=true)

    atomic_write(path) do io
        println(io, join([
            "rank", "candidate_id", "is_facet", "face_dimension", "lp_kinds",
            "best_sample_id", "bound", "best_target_value", "best_normalized_gap",
            "best_critical_visibility", "best_critical_noise_fraction",
        ], '\t'))
        for (rank_index, candidate) in enumerate(ranking)
            println(io, join([
                string(rank_index), candidate["candidate_id"], string(candidate["is_facet"]),
                string(candidate["face_dimension"]), join(candidate["lp_kinds"], ","),
                string(candidate["best_sample_id"]), string(candidate["bound"]),
                float_string(candidate["best_target_value"]),
                float_string(candidate["best_normalized_gap"]),
                float_string(candidate["best_critical_visibility"]),
                float_string(candidate["best_critical_noise_fraction"]),
            ], '\t'))
        end
    end
end

write_ranking_table (generic function with 1 method)

In [23]:
function write_realization_tables(paths, samples)
    atomic_write(paths["states"]) do io
        println(io, "sample_id\tbasis_index\ta\tb\tc\treal\timag")
        for sample in samples, (a, b, c) in TRIPLES
            index = state_index(a, b, c)
            amplitude = sample["state"][index]
            println(io, join([
                string(sample["sample_id"]), string(index), string(a), string(b), string(c),
                float_string(real(amplitude)), float_string(imag(amplitude)),
            ], '\t'))
        end
    end

    atomic_write(paths["measurements"]) do io
        println(io, "sample_id\tparty\tinput\toutput\trow\tcolumn\treal\timag")
        for sample in samples, party in 1:3, input in 0:1, output in 0:1, row in 1:2, column in 1:2
            entry = sample["measurements"][party][input + 1][output + 1][row, column]
            println(io, join([
                string(sample["sample_id"]), string(party), string(input), string(output),
                string(row), string(column), float_string(real(entry)), float_string(imag(entry)),
            ], '\t'))
        end
    end

    atomic_write(paths["behaviors"]) do io
        println(io, "sample_id\tx\ty\tz\ta\tb\tc\tprobability")
        for sample in samples, (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
            probability = sample["behavior"][prob_index(a, b, c, x, y, z)]
            println(io, join([
                string(sample["sample_id"]), string(x), string(y), string(z),
                string(a), string(b), string(c), float_string(probability),
            ], '\t'))
        end
    end
end


function write_halton_table(path,samples)
    atomic_write(path) do io
        headers=["sample_id","sequence_index","shift_seed"]
        append!(headers,["halton_"*s for s in ["u_A","v_A","u_B","v_B","u_C","v_C"]])
        for p in ["A","B","C"];append!(headers,["theta_"*p,"phi_"*p,"nx_"*p,"ny_"*p,"nz_"*p]);end
        println(io,join(headers,'\t'))
        for s in samples
            fields=String[string(s["sample_id"]),string(s["sequence_index"]),string(MASTER_SEED)]
            append!(fields,float_string.(s["halton_point"]))
            for p in 1:3
                append!(fields,float_string.(s["angles"][p,:]))
                append!(fields,float_string.(s["directions"][p,:]))
            end
            println(io,join(fields,'\t'))
        end
    end
end

function write_metadata(path,state,vertex_fingerprint)
    atomic_write(path) do io
        println(io,"Canonical GHZ, input-0 observable Z, shifted-Halton search")
        println(io,"saved_at = ",now())
        println(io,"julia_version = ",VERSION)
        println(io,"JuMP_version = ",Base.pkgversion(JuMP))
        println(io,"MathOptInterface_version = ",Base.pkgversion(MOI))
        println(io,"GLPK_julia_package_version = ",Base.pkgversion(GLPK))
        for key in sort(collect(keys(state["config"])));println(io,key," = ",state["config"][key]);end
        println(io,"halton_shift_17digit = ",join(float_string.(HALTON_SHIFT),","))
        println(io,"completed_samples = ",length(state["samples"]))
        println(io,"next_sample_id = ",state["next_sample_id"])
        println(io,"lp_runs = ",length(state["lp_runs"]))
        println(io,"unique_candidates = ",length(state["candidates"]))
        println(io,"failures = ",length(state["failures"]))
        println(io,"checkpoint_every = ",CHECKPOINT_EVERY)
        println(io,"write_tables_every = ",WRITE_TABLES_EVERY)
        println(io,"fixed_beta_lp = min c.q; c.v>=1; c>=0")
        println(io,"free_beta_lp = min c.q-beta; c.v>=beta; c.u-beta=1; c,beta free")
        println(io,"ranking_scope = candidate_ranking.tsv is discovery-only; final orbit_ranking.tsv cross-evaluates every sampled target and every physical copy")
        println(io,"orbit_ranking_coverage = see orbit_metadata.txt and targets_cross_evaluated in orbit_ranking.tsv")
    end
end

function write_all_tables!(paths,state,vertex_fingerprint)
    write_sample_table(paths["samples"],state["samples"])
    write_lp_run_table(paths["lp_runs"],state["lp_runs"])
    write_candidate_table(paths["candidates"],state["candidates"])
    write_ranking_table(paths["ranking"],state["candidates"])
    write_realization_tables(paths,state["samples"])
    write_halton_table(joinpath(dirname(paths["samples"]),"halton_parameters.tsv"),state["samples"])
    write_metadata(paths["metadata"],state,vertex_fingerprint)
    nothing
end

write_all_tables! (generic function with 1 method)

## 10. Deterministic sampler, both LPs and transactional search engine

State and measurement audits precede the two LP solves. Every target is fixed by its Halton index; restarting never draws an extra shift or skips a point. Keep STOP_ON_ERROR=true for the research run so failures are resolved before proceeding.

In [25]:
function vertex_fingerprint(vertices)
    bytes2hex(sha256(reinterpret(UInt8,vec(copy(vertices)))))
end
const VERTEX_FINGERPRINT=vertex_fingerprint(vertices_cycle)
@assert VERTEX_FINGERPRINT=="93671de2a20e8f3639d08ba6665ec184ca3f58103ff394256a35386df12834ef"

function search_config()
    Dict{String,Any}(
        "sampler"=>"canonical_ghz_shifted_halton_v1","master_seed"=>MASTER_SEED,
        "n_samples"=>N_SAMPLES,"halton_start_index"=>HALTON_START_INDEX,
        "halton_bases"=>HALTON_BASES,"halton_shift"=>HALTON_SHIFT,
        "state"=>"(000+111)/sqrt(2), phase=0","input0_observable"=>"Z for A,B,C",
        "rational_tolerance"=>RATIONAL_RECONSTRUCTION_TOLERANCE,
        "lp_consistency_tolerance"=>LP_CONSISTENCY_TOLERANCE,
        "quantum_audit_tolerance"=>QUANTUM_AUDIT_TOLERANCE,
        "normalized_violation_tolerance"=>NORMALIZED_VIOLATION_TOLERANCE,
        "vertex_fingerprint"=>VERTEX_FINGERPRINT,"coordinate_order"=>"1+c+2b+4a+8z+16y+32x",
    )
end

function fresh_state()
    Dict{String,Any}("format_version"=>2,"config"=>search_config(),"next_sample_id"=>1,
        "samples"=>Any[],"lp_runs"=>Any[],"candidates"=>Any[],
        "candidate_index"=>Dict{String,Int}(),"failures"=>Any[])
end

function validate_checkpoint(state)
    state["format_version"]==2 || error("Checkpoint format mismatch")
    state["config"]==search_config() || error("Checkpoint configuration mismatch; use the original configuration or a fresh output directory")
    length(state["lp_runs"])==2length(state["samples"]) || error("Checkpoint LP count mismatch")
    @assert all(s["sample_id"]<state["next_sample_id"] for s in state["samples"])
    @assert issorted([s["sample_id"] for s in state["samples"]])
    @assert length(unique(s["sample_id"] for s in state["samples"]))==length(state["samples"])
    for s in state["samples"]
        @assert s["sequence_index"]==HALTON_START_INDEX+s["sample_id"]-1
        @assert s["halton_point"]==shifted_halton_point(s["sequence_index"])
    end
    true
end

function sample_spec(sample_id)
    1<=sample_id<=N_SAMPLES || error("Sample outside configured range")
    "canonical_ghz_halton",HALTON_START_INDEX+sample_id-1,MASTER_SEED
end

function generate_sample(sample_id)
    ensemble,index,seed=sample_spec(sample_id)
    psi,M,point,angles,directions=canonical_ghz_realization(index)
    q,imaginary=quantum_behavior(psi,M)
    audit=audit_quantum_realization(psi,M,q,imaginary)
    analytic=analytic_ghz_behavior(M); discrepancy=maximum(abs.(q-analytic))
    discrepancy<=QUANTUM_AUDIT_TOLERANCE || error("Independent GHZ formula audit failed")
    audit["analytic_ghz_discrepancy"]=discrepancy
    @assert psi==CANONICAL_GHZ
    @assert all(M[p][1]==COMPUTATIONAL_MEASUREMENT for p in 1:3)
    Dict{String,Any}("sample_id"=>sample_id,"ensemble"=>ensemble,"ensemble_index"=>index,
        "sequence_index"=>index,"seed"=>seed,"state"=>psi,"measurements"=>M,
        "behavior"=>q,"audit"=>audit,"halton_point"=>point,"angles"=>angles,"directions"=>directions)
end

function solve_both_lps(sample)
    target = sample["behavior"]

    fixed_numerical = solve_fixed_beta_target!(
        fixed_beta_model,
        fixed_beta_coefficients,
        target,
    )
    fixed_outcome = audit_lp_solution(
        "fixed_beta",
        fixed_numerical,
        target,
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise,
    )

    free_numerical = solve_free_beta_target!(
        free_beta_model,
        free_beta_coefficients,
        free_beta_beta,
        target,
    )
    free_outcome = audit_lp_solution(
        "free_beta",
        free_numerical,
        target,
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise,
    )

    return [fixed_outcome, free_outcome]
end

solve_both_lps (generic function with 1 method)

In [26]:
function target_metrics_for_representative(candidate, target)
    target_value = sum(
        Float64(candidate["coefficients"][j]) * target[j]
        for j in eachindex(target)
    )
    gap = Float64(candidate["bound"]) - target_value
    normalized_gap = gap / Float64(candidate["uniform_slack"])
    target_violates = normalized_gap > NORMALIZED_VIOLATION_TOLERANCE
    critical_visibility = target_violates ? 1 / (1 + normalized_gap) : NaN
    critical_noise_fraction = target_violates ? normalized_gap / (1 + normalized_gap) : NaN
    return (
        target_value=target_value,
        gap=gap,
        normalized_gap=normalized_gap,
        critical_visibility=critical_visibility,
        critical_noise_fraction=critical_noise_fraction,
    )
end

function update_unique_candidate!(state, sample, outcome)
    candidate = outcome["candidate"]
    candidate === nothing && return "", false

    key = candidate["canonical_slack_key"]
    index = get(state["candidate_index"], key, 0)
    is_duplicate = index != 0

    if !is_duplicate
        stored = deepcopy(candidate)
        metrics = target_metrics_for_representative(stored, sample["behavior"])
        stored["first_sample_id"] = sample["sample_id"]
        stored["best_sample_id"] = sample["sample_id"]
        stored["lp_kinds"] = String[outcome["lp_kind"]]
        stored["discovery_count"] = 1
        stored["best_target_value"] = metrics.target_value
        stored["best_gap"] = metrics.gap
        stored["best_normalized_gap"] = metrics.normalized_gap
        stored["best_critical_visibility"] = metrics.critical_visibility
        stored["best_critical_noise_fraction"] = metrics.critical_noise_fraction
        push!(state["candidates"], stored)
        index = length(state["candidates"])
        state["candidate_index"][key] = index
    else
        stored = state["candidates"][index]
        metrics = target_metrics_for_representative(stored, sample["behavior"])
        relative_agreement(metrics.normalized_gap, outcome["normalized_gap"]) || error(
            "Gauge-equivalent representatives disagree on normalized target slack",
        )
        stored["discovery_count"] += 1
        if !(outcome["lp_kind"] in stored["lp_kinds"])
            push!(stored["lp_kinds"], outcome["lp_kind"])
            sort!(stored["lp_kinds"])
        end
        if metrics.normalized_gap > stored["best_normalized_gap"]
            stored["best_sample_id"] = sample["sample_id"]
            stored["best_target_value"] = metrics.target_value
            stored["best_gap"] = metrics.gap
            stored["best_normalized_gap"] = metrics.normalized_gap
            stored["best_critical_visibility"] = metrics.critical_visibility
            stored["best_critical_noise_fraction"] = metrics.critical_noise_fraction
        end
    end

    return state["candidates"][index]["candidate_id"], is_duplicate
end

function make_lp_run_record(sample, outcome, candidate_id, is_duplicate)
    candidate = outcome["candidate"]
    return Dict{String, Any}(
        "sample_id" => sample["sample_id"],
        "ensemble" => sample["ensemble"],
        "seed" => sample["seed"],
        "lp_kind" => outcome["lp_kind"],
        "status" => outcome["status"],
        "solver_objective" => outcome["solver_objective"],
        "outcome_kind" => outcome["outcome_kind"],
        "candidate_id" => candidate_id,
        "is_duplicate" => is_duplicate,
        "classical_bound" => candidate === nothing ? "" : string(candidate["bound"]),
        "target_value" => outcome["target_value"],
        "gap" => outcome["gap"],
        "uniform_slack" => candidate === nothing ? "" : rational_string(candidate["uniform_slack"]),
        "normalized_gap" => outcome["normalized_gap"],
        "target_violates" => outcome["target_violates"],
        "critical_visibility" => outcome["critical_visibility"],
        "critical_noise_fraction" => outcome["critical_noise_fraction"],
        "saturation_count" => candidate === nothing ? 0 : candidate["saturation_count"],
        "face_dimension" => candidate === nothing ? -1 : candidate["face_dimension"],
        "is_facet" => candidate === nothing ? false : candidate["is_facet"],
    )
end


"""Stage candidate updates before mutating the persistent sample prefix."""
function commit_sample!(state,sample,outcomes)
    @assert length(outcomes)==2
    @assert Set(o["lp_kind"] for o in outcomes)==Set(["fixed_beta","free_beta"])
    staged=Dict{String,Any}("candidates"=>copy(state["candidates"]),"candidate_index"=>copy(state["candidate_index"]))
    records=Any[]
    for o in outcomes
        c=o["candidate"]
        if c!==nothing
            index=get(staged["candidate_index"],c["canonical_slack_key"],0)
            if index!=0;staged["candidates"][index]=deepcopy(staged["candidates"][index]);end
        end
        id,duplicate=update_unique_candidate!(staged,sample,o)
        push!(records,make_lp_run_record(sample,o,id,duplicate))
    end
    state["candidates"]=staged["candidates"];state["candidate_index"]=staged["candidate_index"]
    push!(state["samples"],sample);append!(state["lp_runs"],records)
    state["next_sample_id"]=sample["sample_id"]+1
    nothing
end

commit_sample!

In [27]:
function run_blind_search!(;
    output_directory=SEARCH_OUTPUT_DIRECTORY,
    resume=RESUME_SEARCH,
    stop_on_error=STOP_ON_ERROR,
    write_tables_every=WRITE_TABLES_EVERY,
    max_new_samples=typemax(Int),
)
    mkpath(output_directory)
    paths = output_paths(output_directory)

    state = if resume && isfile(paths["checkpoint"])
        loaded = read_checkpoint(paths["checkpoint"])
        validate_checkpoint(loaded)
        println(
            "Resuming at sample ", loaded["next_sample_id"],
            " after ", length(loaded["samples"]), " completed samples.",
        )
        loaded
    else
        if any(isfile(path) for path in values(paths))
            error(
                "No usable checkpoint is selected, but the output directory already contains results. " *
                "Choose a new output directory to avoid overwriting a research run.",
            )
        end
        fresh_state()
    end

    total_samples = N_SAMPLES
    started_at = time()
    processed_this_call=0

    try
        while state["next_sample_id"] <= total_samples && processed_this_call<max_new_samples
            sample_id = state["next_sample_id"]
            ensemble, ensemble_index, seed = sample_spec(sample_id)
            verbose_sample = sample_id==1 || sample_id%PRINT_EVERY==0 || sample_id==total_samples
            verbose_sample && println("Sample $sample_id/$total_samples | Halton index $ensemble_index")

            try
                # No persistent state is mutated until generation, both solves,
                # and all exact audits have succeeded.
                sample = generate_sample(sample_id)
                outcomes = solve_both_lps(sample)
                commit_sample!(state, sample, outcomes)

                processed_this_call+=1
                if verbose_sample
                for outcome in outcomes
                    label = outcome["outcome_kind"] == "affine_equality" ?
                        "affine equality" :
                        (outcome["candidate"]["is_facet"] ? "facet" : "nonfacet")
                    @printf(
                        "  %-10s | %-15s | normalized gap = % .9e | noise = %s\n",
                        outcome["lp_kind"],
                        label,
                        outcome["normalized_gap"],
                        float_string(outcome["critical_noise_fraction"]),
                    )
                end

                end # progress output
                if sample_id%CHECKPOINT_EVERY==0
                    write_checkpoint(paths["checkpoint"],state)
                end
                if sample_id % write_tables_every == 0 || sample_id == total_samples
                    write_all_tables!(paths, state, VERTEX_FINGERPRINT)
                end
            catch error_value
                error_value isa InterruptException && rethrow()
                failure = Dict{String, Any}(
                    "timestamp" => string(now()),
                    "sample_id" => sample_id,
                    "ensemble" => ensemble,
                    "stage" => "generate_solve_or_audit",
                    "message" => sprint(showerror, error_value),
                )
                push!(state["failures"], failure)
                append_failure(
                    paths["failures"],
                    sample_id,
                    ensemble,
                    failure["stage"],
                    error_value,
                )
                write_checkpoint(paths["checkpoint"], state)

                if stop_on_error
                    rethrow()
                else
                    @warn "Skipping failed sample" sample_id exception=(error_value, catch_backtrace())
                    state["next_sample_id"] = sample_id + 1
                    write_checkpoint(paths["checkpoint"], state)
                end
            end
        end
    finally
        write_checkpoint(paths["checkpoint"], state)
        write_all_tables!(paths, state, VERTEX_FINGERPRINT)
    end

    elapsed_minutes = (time() - started_at) / 60
    violating_runs = count(run -> run["target_violates"], state["lp_runs"])
    violating_candidates = count(
        candidate -> candidate["best_normalized_gap"] > NORMALIZED_VIOLATION_TOLERANCE,
        state["candidates"],
    )
    facet_candidates = count(candidate -> candidate["is_facet"], state["candidates"])

    println(state["next_sample_id"]>N_SAMPLES ? "\nBlind batch complete" : "\nCommitted prefix saved; run the search cell again to continue")
    println("  completed quantum samples: ", length(state["samples"]))
    println("  LP runs:                   ", length(state["lp_runs"]))
    println("  target-violating LP runs:  ", violating_runs)
    println("  distinct inequalities:     ", length(state["candidates"]))
    println("  distinct facets:            ", facet_candidates)
    println("  violating inequalities:     ", violating_candidates)
    @printf("  elapsed this call:          %.2f minutes\n", elapsed_minutes)
    println("  results directory:          ", abspath(output_directory))

    return state
end

run_blind_search! (generic function with 1 method)

## 11. Run or resume the batch

This cell runs the 10,000-target design unless you changed N_SAMPLES for the installation check. Interrupting preserves the completed prefix. Re-run this cell to continue; restart the kernel and run the preceding cells first if the kernel itself was restarted.

The search does not stop or adapt when it recognizes a useful witness. NPA, see-saws, old witness reference checks and adaptive optimization are outside this blind discovery stage.

In [29]:
search_state = run_blind_search!()

Sample 1/10000 | Halton index 1
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | nonfacet        | normalized gap = -2.960594732e-16 | noise = 
Sample 100/10000 | Halton index 100
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | facet           | normalized gap = -1.578983857e-15 | noise = 
Sample 200/10000 | Halton index 200
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | facet           | normalized gap =  7.401486831e-17 | noise = 
Sample 300/10000 | Halton index 300
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | facet           | normalized gap =  2.220446049e-16 | noise = 
Sample 400/10000 | Halton index 400
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  free_beta  | facet           | normalized gap =  1.973729822e-16 | noise = 
Sample 500/10000 | Halton index 500
  fixed_beta | affine equality | normalized gap =  NaN | noise = 
  

Dict{String, Any} with 8 entries:
  "format_version"  => 2
  "candidate_index" => Dict("1,4,1,4,1,4,1,4,0,3,0,3,0,3,0,3,4,6,4,6,4,6,4,6,3,…
  "config"          => Dict{String, Any}("vertex_fingerprint"=>"93671de2a20e8f3…
  "next_sample_id"  => 10001
  "samples"         => Any[Dict{String, Any}("sample_id"=>1, "audit"=>Dict{Stri…
  "lp_runs"         => Any[Dict{String, Any}("outcome_kind"=>"affine_equality",…
  "failures"        => Any[]
  "candidates"      => Any[Dict{String, Any}("best_critical_visibility"=>NaN, "…

## 12. Exact symmetry-orbit classification

The group contains all 6 party permutations, 8 input-swap choices, and 64 setting-dependent output-flip choices. Its 3,072 maps are distinct, and its 11 generators are checked to preserve the exact vertex set. This verifies invariance of every composition. The group acts on **inequalities** for classification and ranking; it does not alter the restricted target ensemble.

An exact 51-point affine basis is sufficient to identify an affine functional on the 50-dimensional cycle polytope. Each signature is divided by the gcd of the full integer vertex slacks, and every physical image is examined. This identifies affine-gauge and positive-scale equivalence without changing either LP. Full exact signature tuples determine equality; SHA-256 is only the readable family ID.

The standard Int64 IDs use the same basis/order and little-endian encoding as this chat's 64-family catalogue, so existing families can be recognized later without seeding the discovery. A labelled BigInt fallback handles unusually large exact coefficients.

In [31]:
"""Greedy affine basis, in the same vertex order used by the previous project audit."""
function exact_affine_basis_indices(vertices)
    p=first(RANK_PRIMES); selected=[1]; basis=Vector{Vector{Int64}}();pivots=Int[]
    for i in 2:size(vertices,1)
        row=mod.(Int64.(vertices[i,:])-Int64.(vertices[1,:]),p)
        for (pivot,b) in zip(pivots,basis)
            factor=row[pivot]
            factor==0 && continue
            for j in 1:length(row)
                row[j]=mod(row[j]-mod_multiply(factor,b[j],p),p)
            end
        end
        pivot=findfirst(!iszero,row)
        if pivot!==nothing
            inverse=mod_power(row[pivot],p-2,p)
            row=[mod_multiply(v,inverse,p) for v in row]
            push!(basis,row);push!(pivots,pivot);push!(selected,i)
            length(basis)==SIGLOCAL_AFFINE_DIMENSION && break
        end
    end
    @assert length(selected)==51
    @assert rank_mod_prime(difference_matrix(vertices[selected,:]),p)==50
    selected
end

const PARTY_PERMUTATIONS=[(1,2,3),(1,3,2),(2,1,3),(2,3,1),(3,1,2),(3,2,1)]
function physical_permutation(party,swaps,mask)
    permutation=Vector{Int}(undef,64)
    for (x,y,z) in TRIPLES,(a,b,c) in TRIPLES
        oldinputs=(x,y,z);oldoutputs=(a,b,c)
        inputs=ntuple(j->xor(oldinputs[party[j]],swaps[j]),3)
        outputs=ntuple(j->xor(oldoutputs[party[j]],(mask>>(2*(j-1)+inputs[j]))&1),3)
        permutation[prob_index(a,b,c,x,y,z)]=prob_index(outputs...,inputs...)
    end
    @assert sort(permutation)==collect(1:64)
    permutation
end

function build_and_verify_physical_group(vertices)
    permutations=[physical_permutation(p,s,m) for p in PARTY_PERMUTATIONS for s in TRIPLES for m in 0:63]
    @assert length(permutations)==3072
    @assert length(Set(Tuple(p) for p in permutations))==3072
    # The parametrization is exactly the local input/output relabeling group,
    # extended by S3 party permutations. Check invariance of its generators.
    # Their compositions therefore preserve the vertex set for all 3072 maps.
    generators=[physical_permutation(p,(0,0,0),0) for p in [(2,1,3),(1,3,2)]]
    for party in 1:3
        swaps=ntuple(j->j==party ? 1 : 0,3)
        push!(generators,physical_permutation((1,2,3),swaps,0))
    end
    for bit in 0:5;push!(generators,physical_permutation((1,2,3),(0,0,0),1<<bit));end
    vertexset=row_set(vertices)
    for permutation in generators
        @assert all(Tuple(v[permutation]) in vertexset for v in eachrow(vertices))
    end
    println("3072 distinct physical relabelings; all 11 generators preserve the exact vertex set.")
    permutations
end

const AFFINE_BASIS_INDICES=exact_affine_basis_indices(vertices_cycle)
const ORBIT_BASIS_SUPPORTS=[findall(!iszero,v) for v in eachrow(vertices_cycle[AFFINE_BASIS_INDICES,:])]
const PHYSICAL_PERMUTATIONS=build_and_verify_physical_group(vertices_cycle)

function normalized_basis_signature(C,B,divisor,permutation=collect(1:64))
    Tuple(begin
        s=sum((BigInt(C[permutation[j]]) for j in support);init=BigInt(0))-B
        s>=0 && rem(s,divisor)==0 || error("Invalid normalized symmetry signature")
        div(s,divisor)
    end for support in ORBIT_BASIS_SUPPORTS)
end

function byte_less(a,b)
    for i in 1:min(length(a),length(b))
        a[i]==b[i] || return a[i]<b[i]
    end
    length(a)<length(b)
end

"""Explicit little-endian Int64 encoding, compatible with the prior review IDs."""
function signature_bytes(signature)
    result=UInt8[];sizehint!(result,8length(signature))
    for x in signature
        0<=x<=typemax(Int64) || error("Signature does not fit the Int64 encoding")
        v=UInt64(x)
        for shift in 0:8:56;push!(result,UInt8((v>>shift)&0xff));end
    end
    result
end

function classify_physical_orbits(state)
    families=Any[]; signature_family=Dict{Tuple,Int}();membership=Dict{String,String}()
    for (index,candidate) in enumerate(state["candidates"])
        C=candidate["coefficients"];B=candidate["bound"]
        values=integer_vertex_values(vertices_cycle,C);slacks=values.-B
        divisor=foldl(gcd,slacks;init=BigInt(0));@assert divisor>0
        key=normalized_basis_signature(C,B,divisor)
        fi=get(signature_family,key,0)
        if fi==0
            seen=Set{Tuple}();copyindices=Int[];signatures=Tuple[]
            int64encoding=div(maximum(slacks),divisor)<=typemax(Int64)
            canonical_bytes=nothing;canonical_text=nothing
            for (pi,perm) in enumerate(PHYSICAL_PERMUTATIONS)
                signature=normalized_basis_signature(C,B,divisor,perm)
                signature in seen && continue
                push!(seen,signature);push!(copyindices,pi);push!(signatures,signature)
                if int64encoding
                    bytes=signature_bytes(signature)
                    if canonical_bytes===nothing || byte_less(bytes,canonical_bytes);canonical_bytes=bytes;end
                else
                    text=join(signature,",")
                    if canonical_text===nothing || isless(text,canonical_text);canonical_text=text;end
                end
            end
            familyid=int64encoding ? bytes2hex(sha256(canonical_bytes)) : "bigint_"*bytes2hex(sha256(codeunits(canonical_text)))
            stored=Dict{String,Any}(
                "family_id"=>familyid,"representative"=>candidate,
                "copy_indices"=>copyindices,"member_candidate_ids"=>String[],
                "first_sample_id"=>candidate["first_sample_id"],"lp_kinds"=>String[],
                "encoding"=>int64encoding ? "little_endian_int64_v1" : "bigint_decimal_v1",
            )
            push!(families,stored);fi=length(families)
            for signature in signatures
                haskey(signature_family,signature) && error("Physical orbit overlap")
                signature_family[signature]=fi
            end
        end
        f=families[fi]
        @assert f["representative"]["face_dimension"]==candidate["face_dimension"]
        push!(f["member_candidate_ids"],candidate["candidate_id"])
        f["first_sample_id"]=min(f["first_sample_id"],candidate["first_sample_id"])
        append!(f["lp_kinds"],candidate["lp_kinds"]);unique!(f["lp_kinds"]);sort!(f["lp_kinds"])
        membership[candidate["candidate_id"]]=f["family_id"]
        if index%25==0;println("Orbit classification: ",index,"/",length(state["candidates"])," candidates; ",length(families)," families");end
    end
    println("Exact physical families: ",length(families)," from ",length(state["candidates"])," exact candidates.")
    families,membership
end

function rank_physical_orbits(state,families;output_directory=SEARCH_OUTPUT_DIRECTORY,blocksize=CROSS_EVALUATION_BLOCK_SIZE)
    samples=state["samples"];N=length(samples);@assert N>0
    Q=hcat([s["behavior"] for s in samples]...)
    ranked=Any[]
    for (fi,f) in enumerate(families)
        r=f["representative"];C=r["coefficients"];B=r["bound"];W=r["uniform_slack"]
        copies=[C[PHYSICAL_PERMUTATIONS[pi]] for pi in f["copy_indices"]]
        normalized=hcat([Float64.(c)./Float64(W) for c in copies]...)'
        bestgap=-Inf;bestcopy=0;bestsample=0
        for start in 1:blocksize:N
            finish=min(N,start+blocksize-1)
            gaps=Float64(B/W).-normalized*view(Q,:,start:finish)
            g,where=findmax(gaps)
            if g>bestgap
                bestgap=g;bestcopy=where[1];bestsample=start+where[2]-1
            end
        end
        # Independently reconstruct the maximum using the integer witness.
        bestC=copies[bestcopy];q=samples[bestsample]["behavior"]
        Iq=sum(Float64(bestC[j])*q[j] for j in 1:64)
        directgap=(Float64(B)-Iq)/Float64(W)
        abs(bestgap-directgap)<=1e-9*(1+abs(directgap)) || error("Cross-evaluation consistency failure")
        violation=directgap>NORMALIZED_VIOLATION_TOLERANCE
        row=copy(f);merge!(row,Dict{String,Any}(
            "best_sample_id"=>samples[bestsample]["sample_id"],"best_sample_position"=>bestsample,
            "best_permutation_index"=>f["copy_indices"][bestcopy],
            "best_coefficients"=>bestC,"best_target_value"=>Iq,"best_normalized_gap"=>directgap,
            "target_violates"=>violation,"best_critical_visibility"=>violation ? 1/(1+directgap) : NaN,
            "best_critical_noise_fraction"=>violation ? directgap/(1+directgap) : NaN,
            "targets_cross_evaluated"=>N,"physical_copies_cross_evaluated"=>length(copies),
        ))
        push!(ranked,row)
        println("Symmetry cross-evaluation ",fi,"/",length(families)," | ",first(f["family_id"],12)," | best noise = ",float_string(row["best_critical_noise_fraction"]))
    end
    sort!(ranked;by=f->f["best_normalized_gap"],rev=true)
    tie_group=0;anchor=Inf
    for f in ranked
        if tie_group==0 || abs(f["best_normalized_gap"]-anchor)>RANKING_TIE_TOLERANCE
            tie_group+=1;anchor=f["best_normalized_gap"]
        end
        f["numerical_tie_group"]=tie_group
    end
    write_orbit_outputs(output_directory,state,ranked)
    ranked
end

function write_orbit_outputs(directory,state,ranked)
    mkpath(directory)
    atomic_write(joinpath(directory,"orbit_ranking.tsv")) do io
        headers=["rank","numerical_tie_group","family_id","representative_candidate_id","is_facet","face_dimension","saturation_count","first_sample_id","lp_kinds","stored_candidate_count","physical_orbit_size","targets_cross_evaluated","best_sample_id","best_sequence_index","best_permutation_index","bound","uniform_slack","best_target_value","best_normalized_gap","target_violates","best_critical_visibility","best_critical_noise_fraction","best_noise_percent","best_coefficients_in_canonical_64_order","representative_coefficients_in_canonical_64_order"]
        println(io,join(headers,'\t'))
        for (i,f) in enumerate(ranked)
            r=f["representative"];s=state["samples"][f["best_sample_position"]]
            values=Any[i,f["numerical_tie_group"],f["family_id"],r["candidate_id"],r["is_facet"],r["face_dimension"],r["saturation_count"],f["first_sample_id"],join(f["lp_kinds"],","),length(f["member_candidate_ids"]),length(f["copy_indices"]),length(state["samples"]),f["best_sample_id"],s["sequence_index"],f["best_permutation_index"],r["bound"],rational_string(r["uniform_slack"]),float_string(f["best_target_value"]),float_string(f["best_normalized_gap"]),f["target_violates"],float_string(f["best_critical_visibility"]),float_string(f["best_critical_noise_fraction"]),float_string(100*f["best_critical_noise_fraction"]),integer_vector_string(f["best_coefficients"]),integer_vector_string(r["coefficients"])]
            println(io,join(string.(values),'\t'))
        end
    end
    atomic_write(joinpath(directory,"orbit_membership.tsv")) do io
        println(io,"candidate_id\tfamily_id\tface_dimension\tis_facet")
        for f in ranked,id in f["member_candidate_ids"]
            println(io,join([id,f["family_id"],string(f["representative"]["face_dimension"]),string(f["representative"]["is_facet"])],'\t'))
        end
    end
    atomic_write(joinpath(directory,"bell_relabelings.tsv")) do io
        println(io,"permutation_index\tcoefficient_copy_j_equals_original_at_index")
        for (i,p) in enumerate(PHYSICAL_PERMUTATIONS);println(io,string(i),'\t',join(p,","));end
    end
    atomic_write(joinpath(directory,"orbit_metadata.txt")) do io
        println(io,"physical_group_size = 3072")
        println(io,"affine_basis_vertex_indices_one_based = ",join(AFFINE_BASIS_INDICES,","))
        println(io,"affine_basis_rank_mod_prime = 50")
        println(io,"basis_rank_prime = ",first(RANK_PRIMES))
        println(io,"completed_targets_scored = ",length(state["samples"]))
        println(io,"planned_targets = ",N_SAMPLES)
        println(io,"complete_planned_batch = ",state["next_sample_id"]>N_SAMPLES)
        println(io,"families = ",length(ranked))
        println(io,"numerical_tie_tolerance_normalized_gap = ",RANKING_TIE_TOLERANCE)
        println(io,"id_rule = SHA256 of lexicographically least 51-point normalized slack byte string over the 3072 relabelings; Int64 little-endian; BigInt decimal fallback labelled separately")
        println(io,"equality_rule = full exact normalized basis tuples, not SHA256 alone")
        println(io,"ranking_scope = every stored target times every physically distinct copy of every discovered family")
        println(io,"best_witness_coefficients_match_original_saved_quantum_sample = true")
    end
end

3072 distinct physical relabelings; all 11 generators preserve the exact vertex set.


write_orbit_outputs (generic function with 1 method)

In [32]:
physical_families, candidate_to_family = classify_physical_orbits(search_state)

Orbit classification: 25/110 candidates; 8 families
Orbit classification: 50/110 candidates; 9 families
Orbit classification: 75/110 candidates; 11 families
Orbit classification: 100/110 candidates; 16 families
Exact physical families: 18 from 110 exact candidates.


(Any[Dict{String, Any}("member_candidate_ids" => ["495fc65a991a6f678f6f1e3267f136cc1b684cb224497d647b6d37e39c3aba32"], "family_id" => "1e3085dac682463d57df6697aa6ad5b01b35ca3f5fbd75525f20dd29b7fdd7c2", "lp_kinds" => ["free_beta"], "copy_indices" => [1, 2, 5, 6, 65, 66, 69, 70, 129, 130  …  325, 326, 385, 386, 389, 390, 449, 450, 453, 454], "first_sample_id" => 1, "encoding" => "little_endian_int64_v1", "representative" => Dict{String, Any}("best_critical_visibility" => NaN, "best_gap" => -2.220446049250313e-16, "best_sample_id" => 1, "best_critical_noise_fraction" => NaN, "bound" => -1, "counterclockwise_minimum" => -1, "face_dimension" => 44, "lp_kinds" => ["free_beta"], "candidate_id" => "495fc65a991a6f678f6f1e3267f136cc1b684cb224497d647b6d37e39c3aba32", "clockwise_minimum" => -1…)), Dict{String, Any}("member_candidate_ids" => ["ffd193883a464ceb9f98721d59f471c2afb0c8567b2871b754e7188b44836315", "21e69dddffcd7a457c94483ab5054a2baf894e607290da44b5674c5b1add41ad", "18101164144954240e771

## 13. Rank every family over every target and every physical copy

All exact candidates participate, even those not violated by their generating target. The block calculation evaluates every physically distinct copy on every saved target; it does not rank a family only on the samples that returned it. The best row stores the actual integer coefficient copy, bound and original sample ID. Its saved state and measurements therefore match that witness directly, with no implicit relabeling of the realization.

For I>=B, define W=I(u)-B and G=(B-I(q))/W. Then

$$\eta_{crit}=\frac{G}{1+G},\qquad v_{crit}=\frac1{1+G}.$$

White-noise fraction eta means the fraction in $(1-\eta)q+\eta u$. Larger positive eta is better. A nonviolated family has a blank noise entry, rather than a negative noise tolerance. This is the best **sampled** tolerance for a witness; it is not an optimized quantum value or a proof of polytope membership above the threshold. Facets and nonfacets are both retained in orbit_ranking.tsv.

`numerical_tie_group` marks normalized gaps within RANKING_TIE_TOLERANCE of the group's leading row. Rows in the same group should be read as numerical ties; tiny differences in their file order do not establish superiority.

In [34]:
orbit_ranking = rank_physical_orbits(search_state, physical_families)

Symmetry cross-evaluation 1/18 | 1e3085dac682 | best noise = 
Symmetry cross-evaluation 2/18 | 30f4b3855227 | best noise = 0.0036616893197959112
Symmetry cross-evaluation 3/18 | 9397789ba938 | best noise = 0.0036616893197956497
Symmetry cross-evaluation 4/18 | f25ba931224d | best noise = 0.0036616893197959112
Symmetry cross-evaluation 5/18 | 0512b86fa82c | best noise = 
Symmetry cross-evaluation 6/18 | 0cfed64be21c | best noise = 
Symmetry cross-evaluation 7/18 | ad9abbd54d00 | best noise = 
Symmetry cross-evaluation 8/18 | d73245df1e67 | best noise = 
Symmetry cross-evaluation 9/18 | e8a47dc3592d | best noise = 
Symmetry cross-evaluation 10/18 | b6aae966891c | best noise = 
Symmetry cross-evaluation 11/18 | 1fca8dc68cfa | best noise = 
Symmetry cross-evaluation 12/18 | 485f1c8ceff9 | best noise = 
Symmetry cross-evaluation 13/18 | cc898d5d290a | best noise = 
Symmetry cross-evaluation 14/18 | bd9820f54eb7 | best noise = 
Symmetry cross-evaluation 15/18 | 20b5e92818fb | best noise = 
S

18-element Vector{Any}:
 Dict{String, Any}("copy_indices" => [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  503, 504, 505, 506, 507, 508, 509, 510, 511, 512], "best_critical_visibility" => 0.9963383106802041, "physical_copies_cross_evaluated" => 512, "best_sample_id" => 3393, "numerical_tie_group" => 1, "best_coefficients" => BigInt[0, 15, 15, 12, 15, 12, 12, 0, 5, 0  …  10, 0, 5, 0, 0, 4, 0, 4, 4, 8], "best_critical_noise_fraction" => 0.003661689319795911, "best_permutation_index" => 33, "member_candidate_ids" => ["ffd193883a464ceb9f98721d59f471c2afb0c8567b2871b754e7188b44836315", "21e69dddffcd7a457c94483ab5054a2baf894e607290da44b5674c5b1add41ad", "18101164144954240e771b8b7d2eb38b710d2b70c0c36beaad93c3ab0d91ac37", "607583b4df823292eda6950e12c62726dba28e39170a7bd25e5aea61c7f6ed88", "7dcd88c4386ad4108b3f2079d161097c2a7a996e0d87aa2fb47a2a4f8b7b7cfe", "60cdec0aaeff19960d966110a9210d54679c81b65988b725911b2cc45396cd8a", "4598ddb433ac5e87e7e9c11d36d83ff6aa0bf165a2a51784d451dd58a2404480", "fc36019c86f006

In [35]:
println("\nBest symmetry families: achieved sampled white-noise fractions")
println("rank  familyID       facet  face-dim  sample     noise percent")
for (i,f) in enumerate(first(orbit_ranking,min(20,length(orbit_ranking))))
    @printf("%4d  %-12s  %5s  %8d  %6d  %s\n",i,first(f["family_id"],12),string(f["representative"]["is_facet"]),f["representative"]["face_dimension"],f["best_sample_id"],float_string(100f["best_critical_noise_fraction"]))
end
println("Main result: ",joinpath(abspath(SEARCH_OUTPUT_DIRECTORY),"orbit_ranking.tsv"))


Best symmetry families: achieved sampled white-noise fractions
rank  familyID       facet  face-dim  sample     noise percent
   1  30f4b3855227  false        48    3393  0.36616893197959111
   2  f25ba931224d   true        49    3393  0.36616893197959111
   3  9397789ba938   true        49    3393  0.36616893197956496
   4  1ecf9c7a6625  false        48     165  
   5  e8a47dc3592d   true        49    6479  
   6  b6aae966891c  false        48      14  
   7  becc76b83116  false        46    4941  
   8  485f1c8ceff9  false        47    1088  
   9  63c515aa22cb  false        47     787  
  10  cc898d5d290a  false        48    5226  
  11  bd9820f54eb7  false        47     322  
  12  0cfed64be21c  false        48     140  
  13  20b5e92818fb  false        48       2  
  14  ad9abbd54d00   true        49     380  
  15  d73245df1e67   true        49       7  
  16  1fca8dc68cfa  false        48     187  
  17  1e3085dac682  false        44       1  
  18  0512b86fa82c   true        4

## 14. Files to retain and send back

After the full run, zip the entire automatically named result directory. The main result is **orbit_ranking.tsv**. It includes the family ID, face dimension, facet flag, saturation count, discovery LP kinds, number of symmetry copies, best target/sequence index, direct inequality value, noise fraction, and the exact best coefficient vector.

- `search_checkpoint.jls`: resume state and exact candidate records.
- `run_metadata.txt`, `halton_parameters.tsv`, `samples.tsv`: configuration, actual shifts/points/angles and quantum audits.
- `lp_runs.tsv`, `unique_candidates.tsv`, `candidate_ranking.tsv`: both LP outcomes and exact discovery records.
- `quantum_states.tsv`, `quantum_measurements.tsv`, `quantum_behaviors.tsv`: all explicit quantum realizations.
- `orbit_membership.tsv`, `bell_relabelings.tsv`, `orbit_metadata.txt`: family membership, precise relabelings, signature convention and ranking coverage.
- `failures.tsv`, if created: failures to resolve before treating the intended plan as completed.

Check that metadata reports 10,000 completed targets, 20,000 LP records, zero unresolved failures, and complete ranking coverage. A finite low-discrepancy search is a discovery experiment, not exhaustive enumeration of all inequalities or a theorem that no better witness exists.